In [1]:
from bson import decode_file_iter
from collections import defaultdict

import numpy as np
import torch
import matplotlib.pyplot as plt

In [2]:
HEADER_FILE = "bson_file/header.bson"
ACC_FILE = "bson_file/accelerogram.bson"

In [3]:
print("=" * 60)
print("Reading header.bson")
print("=" * 60)

headers = {}

count = 0

with open(HEADER_FILE, "rb") as f:

    for doc in decode_file_iter(f):

        count += 1

        headers[doc["_id"]] = {
            "Station": doc["StationCode"],
            "SamplingRate": doc["SamplingRate_Hz"],
            "PIndex": int(doc["pIndex"]),
            "PGA": doc["PGA_gal"]
        }

print("Headers Loaded :", count)

Reading header.bson
Headers Loaded : 38030


In [4]:
print("=" * 60)
print("Reading accelerogram.bson")
print("=" * 60)

records = []

count = 0

with open(ACC_FILE, "rb") as f:

    for doc in decode_file_iter(f):

        records.append(doc)

        count += 1

print("Signal Records :", count)

Reading accelerogram.bson
Signal Records : 114090


In [25]:
print("=" * 60)
print("Grouping Signals")
print("=" * 60)

from collections import defaultdict

events = defaultdict(dict)

for doc in records:

    full_id = doc["_id"]

    event_id, component = full_id.rsplit(".", 1)

    events[event_id][component] = np.asarray(
        doc["accelerogram"],
        dtype=np.float32
    )
print(events)
print("Total Events :", len(events))

Grouping Signals


KeyboardInterrupt: 

In [26]:
print(events)


defaultdict(<class 'dict'>, {'Botanical Garden_20150206083723': {'UD': array([ 4.1086951e-05,  6.1439365e-05,  2.2431113e-05, ...,
       -2.9203949e-05, -8.4363346e-05, -6.0755476e-05], dtype=float32), 'NS': array([-1.6613758e-05, -1.0700721e-05,  3.3107106e-05, ...,
        7.2485011e-05,  7.9165337e-05,  7.9812766e-05], dtype=float32), 'EW': array([ 2.2864791e-05, -1.1307024e-05, -7.2437347e-06, ...,
       -2.3298015e-05, -2.7188098e-05, -8.7700852e-07], dtype=float32)}, 'Botanical Garden_20150212012151': {'UD': array([-8.0688333e-05,  3.4537036e-06,  6.8636611e-05, ...,
       -3.1021424e-05,  1.2315833e-05,  4.1895459e-05], dtype=float32), 'NS': array([ 7.4596399e-05,  2.0922693e-05, -2.5814626e-05, ...,
       -1.0818707e-04, -7.6987300e-05,  4.2310963e-05], dtype=float32), 'EW': array([-9.8556600e-05, -4.9494160e-05,  2.7859522e-05, ...,
        1.2401078e-04,  1.1010183e-04, -4.8925931e-06], dtype=float32)}, 'Botanical Garden_20150213114109': {'UD': array([ 4.9716960e-05,  5.1

In [6]:
print("=" * 60)
print("Verifying First Complete Event")
print("=" * 60)

for event_id, comp in events.items():

    # Check all three channels exist
    if not all(c in comp for c in ["UD", "NS", "EW"]):
        continue

    # Check header exists
    if event_id not in headers:
        continue

    header = headers[event_id]

    print("Event ID       :", event_id)
    print("Station        :", header["Station"])
    print("Sampling Rate  :", header["SamplingRate"])
    print("PIndex         :", header["PIndex"])
    print("PGA            :", header["PGA"])

    print()

    print("UD Length :", len(comp["UD"]))
    print("NS Length :", len(comp["NS"]))
    print("EW Length :", len(comp["EW"]))

    if len(comp["UD"]) == len(comp["NS"]) == len(comp["EW"]):
        print("\nStatus : ✓ All channels have equal length")
    else:
        print("\nStatus : ✗ Channel lengths are different")

    break

Verifying First Complete Event
Event ID       : Botanical Garden_20150206083723
Station        : Botanical Garden
Sampling Rate  : 100
PIndex         : 1000
PGA            : 0.0003663526935582753

UD Length : 4173
NS Length : 4173
EW Length : 4173

Status : ✓ All channels have equal length


In [7]:
print("=" * 60)
print("Stage 2 : Select One Event")
print("=" * 60)

for event_id, comp in events.items():

    if event_id not in headers:
        continue

    if not all(c in comp for c in ["UD", "NS", "EW"]):
        continue

    header = headers[event_id]

    break

print("Selected Event :", event_id)
print("Station :", header["Station"])
print("Sampling Rate :", header["SamplingRate"])

Stage 2 : Select One Event
Selected Event : Botanical Garden_20150206083723
Station : Botanical Garden
Sampling Rate : 100


In [8]:
print("=" * 60)
print("Stage 2 : Trim Channels")
print("=" * 60)

min_length = min(
    len(comp["UD"]),
    len(comp["NS"]),
    len(comp["EW"])
)

ud = comp["UD"][:min_length]
ns = comp["NS"][:min_length]
ew = comp["EW"][:min_length]

print("Final Length :", min_length)

Stage 2 : Trim Channels
Final Length : 4173


In [9]:
print("=" * 60)
print("Stage 2 : Normalize")
print("=" * 60)

def normalize(signal):
    signal = np.asarray(signal, dtype=np.float32)

    mean = signal.mean()
    std = signal.std()

    if std == 0:
        std = 1.0

    return (signal - mean) / std

ud = normalize(ud)
ns = normalize(ns)
ew = normalize(ew)

print("UD Mean :", round(float(ud.mean()), 5))
print("UD Std  :", round(float(ud.std()), 5))

print("NS Mean :", round(float(ns.mean()), 5))
print("NS Std  :", round(float(ns.std()), 5))

print("EW Mean :", round(float(ew.mean()), 5))
print("EW Std  :", round(float(ew.std()), 5))

Stage 2 : Normalize
UD Mean : 0.0
UD Std  : 1.0
NS Mean : 0.0
NS Std  : 1.0
EW Mean : 0.0
EW Std  : 1.0


In [10]:
print("=" * 60)
print("Stage 2 : Create 2-Second Windows")
print("=" * 60)

WINDOW_SIZE = 200
STEP_SIZE = 200      # non-overlapping windows

windows = []

for start in range(0, min_length - WINDOW_SIZE + 1, STEP_SIZE):

    end = start + WINDOW_SIZE

    window = np.stack([
        ud[start:end],
        ns[start:end],
        ew[start:end]
    ])

    windows.append(window)

windows = np.array(windows)

print("Total Windows :", len(windows))
print("Window Shape :", windows[0].shape)
print("Dataset Shape :", windows.shape)

Stage 2 : Create 2-Second Windows
Total Windows : 20
Window Shape : (3, 200)
Dataset Shape : (20, 3, 200)


In [11]:
print("=" * 60)
print("Stage 3 : Preparing Tensor")
print("=" * 60)

# (20,3,200) -> (20,200,3)
windows = np.transpose(windows, (0, 2, 1))

print("After Transpose :", windows.shape)

windows_tensor = torch.from_numpy(windows).float()

print("Tensor Shape :", windows_tensor.shape)
print("Tensor Type :", windows_tensor.dtype)

Stage 3 : Preparing Tensor
After Transpose : (20, 200, 3)
Tensor Shape : torch.Size([20, 200, 3])
Tensor Type : torch.float32


In [13]:
print("=" * 60)
print("Checking First Window")
print("=" * 60)

print("Shape :", windows_tensor[0].shape)

print("\nFirst 5 UD samples")
print(windows_tensor[0][:5][0])

print("\nFirst 5 NS samples")
print(windows_tensor[0][:5][1])

print("\nFirst 5 EW samples")
print(windows_tensor[0][:5][2])

Checking First Window
Shape : torch.Size([200, 3])

First 5 UD samples
tensor([ 0.9407, -0.2804,  0.3608])

First 5 NS samples
tensor([ 1.4062, -0.1798, -0.1768])

First 5 EW samples
tensor([ 0.5141,  0.5654, -0.1129])


In [40]:
print("=" * 60)
print("Stage 4 : Loading Pretrained Model")
print("=" * 60)

model = TFEQ(channel=3, time_in=200)

checkpoint = torch.load(
    "models/TFEQ.pth",
    map_location="cpu"
)

print("Checkpoint Type :", type(checkpoint))

Stage 4 : Loading Pretrained Model


AssertionError: Torch not compiled with CUDA enabled

In [41]:
checkpoint = torch.load(
    "models/TFEQ.pth",
    map_location="cpu"
)

print(type(checkpoint))

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch.nn.modules.linear.Linear was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch.nn.modules.linear.Linear])` or the `torch.serialization.safe_globals([torch.nn.modules.linear.Linear])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to lea

In [12]:
from utils.TFEQ import TFEQ

In [15]:
import torch

print("Torch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

Torch Version: 2.0.1+cpu
CUDA Available: False


In [13]:
import torch
import torch.nn.modules.linear

# --- THE PATCH ---
# Map the old, deleted internal class to the modern Linear class
torch.nn.modules.linear._LinearWithBias = torch.nn.Linear 
# -----------------

# Now try loading the model
checkpoint = torch.load(
    "models/TFEQ.pth",
    map_location="cpu",
    weights_only=False 
)

print("Checkpoint loaded successfully!")
print("Type:", type(checkpoint))

Checkpoint loaded successfully!
Type: <class 'utils.TFEQ.TFEQ'>


In [14]:
model = checkpoint
model.eval()

print("Model is ready!")

Model is ready!


In [18]:
print(model)

TFEQ(
  (input_embedding): Linear(in_features=3, out_features=64, bias=True)
  (temporal_encoder_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): Linear(in_features=64, out_features=64, bias=True)
    )
    (linear1): Linear(in_features=64, out_features=32, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=32, out_features=64, bias=True)
    (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (temporal_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): Linear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=32, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      

In [19]:
model = checkpoint

state = model.state_dict()

print(type(state))
print("Number of parameters:", len(state))

print("\nFirst 10 parameter names:")
for i, key in enumerate(state.keys()):
    print(key)
    if i == 9:
        break

<class 'collections.OrderedDict'>
Number of parameters: 31

First 10 parameter names:
input_embedding.weight
input_embedding.bias
temporal_encoder_layer.self_attn.in_proj_weight
temporal_encoder_layer.self_attn.in_proj_bias
temporal_encoder_layer.self_attn.out_proj.weight
temporal_encoder_layer.self_attn.out_proj.bias
temporal_encoder_layer.linear1.weight
temporal_encoder_layer.linear1.bias
temporal_encoder_layer.linear2.weight
temporal_encoder_layer.linear2.bias


In [20]:
torch.save(state, "models/TFEQ_state_dict.pth")
print("State dict saved successfully!")

State dict saved successfully!


In [15]:
from utils.TFEQ import TFEQ

model = TFEQ(channel=3, time_in=200)

In [16]:
state = torch.load(
    "models/TFEQ_state_dict.pth",
    map_location="cpu"
)

model.load_state_dict(state)

model.eval()

print("Model Loaded Successfully!")

Model Loaded Successfully!


In [23]:
print(model)

TFEQ(
  (input_embedding): Linear(in_features=3, out_features=64, bias=True)
  (temporal_encoder_layer): TransformerEncoderLayer(
    (self_attn): MultiheadAttention(
      (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
    )
    (linear1): Linear(in_features=64, out_features=32, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (linear2): Linear(in_features=32, out_features=64, bias=True)
    (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    (dropout1): Dropout(p=0.1, inplace=False)
    (dropout2): Dropout(p=0.1, inplace=False)
  )
  (temporal_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=32, bias=True)
     

In [17]:
with torch.no_grad():

    output = model(windows_tensor)

print(output.shape)
print(output[:5])

torch.Size([20, 2])
tensor([[ 0.0000e+00, -2.5153e+01],
        [-4.5299e-06, -1.2294e+01],
        [-3.5763e-06, -1.2539e+01],
        [-5.0181e+00, -6.6391e-03],
        [-1.1892e+01, -6.7949e-06]])


In [18]:
probabilities = torch.exp(output)

print(probabilities[:20])

tensor([[1.0000e+00, 1.1918e-11],
        [1.0000e+00, 4.5787e-06],
        [1.0000e+00, 3.5854e-06],
        [6.6171e-03, 9.9338e-01],
        [6.8419e-06, 9.9999e-01],
        [9.9998e-01, 2.2741e-05],
        [1.0000e+00, 2.2387e-07],
        [6.3925e-05, 9.9994e-01],
        [1.0000e+00, 1.9454e-08],
        [2.9046e-02, 9.7095e-01],
        [1.2754e-03, 9.9872e-01],
        [9.9986e-01, 1.4432e-04],
        [9.9923e-01, 7.7006e-04],
        [1.0000e+00, 4.8914e-06],
        [9.1702e-05, 9.9991e-01],
        [9.9703e-01, 2.9739e-03],
        [4.3020e-02, 9.5698e-01],
        [6.0250e-06, 9.9999e-01],
        [1.0000e+00, 6.2793e-13],
        [9.7853e-01, 2.1466e-02]])


In [23]:
predictions = torch.argmax(probabilities, dim=1)
print(predictions)

tensor([1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1])


In [ ]:
import pandas as pd

results = []

WINDOW_SIZE = 200
STEP_SIZE = 200

for i, (event_id, comp) in enumerate(events.items()):

    # Stop after 100 events
    if i >= 100:
        break

    # Skip incomplete events
    if event_id not in headers:
        continue

    if not all(c in comp for c in ["UD", "NS", "EW"]):
        continue

    # ----------------------------
    # Trim channels
    # ----------------------------
    min_length = min(
        len(comp["UD"]),
        len(comp["NS"]),
        len(comp["EW"])
    )

    ud = comp["UD"][:min_length]
    ns = comp["NS"][:min_length]
    ew = comp["EW"][:min_length]

    # ----------------------------
    # Normalize
    # ----------------------------
    ud = normalize(ud)
    ns = normalize(ns)
    ew = normalize(ew)

    # ----------------------------
    # Create windows
    # ----------------------------
    windows = []

    for start in range(0, min_length - WINDOW_SIZE + 1, STEP_SIZE):

        end = start + WINDOW_SIZE

        window = np.stack([
            ud[start:end],
            ns[start:end],
            ew[start:end]
        ])

        windows.append(window)

    if len(windows) == 0:
        continue

    # ----------------------------
    # Convert to tensor
    # ----------------------------
    windows = np.array(windows)
    windows = np.transpose(windows, (0, 2, 1))
    windows_tensor = torch.from_numpy(windows).float()

    # ----------------------------
    # Prediction
    # ----------------------------
    with torch.no_grad():
        output = model(windows_tensor)

    probabilities = torch.exp(output)
    predictions = torch.argmax(probabilities, dim=1)

    # ----------------------------
    # Save result
    # ----------------------------
    results.append({
        "Event ID": event_id,
        "Window Predictions": predictions.tolist()
    })

# ----------------------------
# Create DataFrame
# ----------------------------
df = pd.DataFrame(results)

print(df)

df.to_csv("first_100_event_predictions.csv", index=False)

print(f"\nProcessed {len(df)} events.")

                           Event ID  \
0   Botanical Garden_20150206083723   
1   Botanical Garden_20150212012151   
2   Botanical Garden_20150213114109   
3   Botanical Garden_20150213115020   
4   Botanical Garden_20150213125121   
..                              ...   
95  Botanical Garden_20150304165220   
96  Botanical Garden_20150304165254   
97  Botanical Garden_20150304170627   
98  Botanical Garden_20150304170756   
99  Botanical Garden_20150304190946   

                                   Window Predictions  
0   [0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, ...  
1   [1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, ...  
2   [0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, ...  
3   [0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, ...  
4   [0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, ...  
..                                                ...  
95  [0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, ...  
96  [0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, ...  
97  [0, 1, 1, 0, 1, 0, 1, 1

In [29]:
min_length = min(
        len(comp["UD"]),
        len(comp["NS"]),
        len(comp["EW"])
    )
print(min_length)

4294
